# Agentic Mental Health Chatbot using LangChain + LangGraph + ChromaDB + Free Hugging Face LLM

**IMPORTANT DISCLAIMER**  
This is an **educational prototype only**. It is **NOT** a substitute for professional mental health care, therapy, or medical advice.  
- The chatbot can make mistakes.  
- If you are in crisis, feeling suicidal, or need immediate help, STOP and contact a professional right now:  
  - India: iCall (9152987821), AASRA (91-9820466726), or dial 112  
  - International: https://findahelpline.com  
- Always consult a licensed therapist or doctor.

---

# Install dependencies

In [1]:
!pip install langchain langgraph langchain-chroma langchain-huggingface langchain-classic
!pip install chromadb sentence-transformers transformers huggingface_hub langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

# Import Dependencies

In [8]:
import os
import json
import logging
from typing import Annotated, TypedDict

# LangChain & LangGraph Imports
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Reranking Imports (Using classic to avoid LangChain 1.0 bugs)
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Transformers Import
from transformers import pipeline

# Setup & Authention

In [ ]:
# ==========================================
# 1. SETUP & AUTHENTICATION
# ==========================================
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

# TODO: Insert your Hugging Face Token here
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""  # Provide the hugging face API

# LLM Initialization

In [10]:
# ==========================================
# 2. LLM INITIALIZATION
# ==========================================
print("⏳ Initializing LLM (GPT-OSS-20b)...")
hf_endpoint = HuggingFaceEndpoint(
    # repo_id="Qwen/Qwen2.5-7B-Instruct",
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    temperature=0.7,
    max_new_tokens=1024,
    return_full_text=False,
    do_sample=True
)
llm = ChatHuggingFace(llm=hf_endpoint)
print("✅ LLM loaded successfully!")

⏳ Initializing LLM (GPT-OSS-20b)...
✅ LLM loaded successfully!


#  Vector Store (Persistent) & Knowledge Base

In [11]:
# ==========================================
# 3. VECTOR STORE (PERSISTENT) & KNOWLEDGE BASE
# ==========================================
print("⏳ Setting up Persistent Knowledge Base...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create a sample complex JSON if it doesn't exist (for testing)
file_path = "mental_health_knowledge.json"
if not os.path.exists(file_path):
    with open(file_path, "w") as f:
        json.dump({
            "knowledge_base": [
                {
                    "topic": "Anxiety",
                    "subtopics": {
                        "coping_techniques": [
                            "Deep breathing exercises: Try 4-7-8 breathing.",
                            "Grounding techniques: 5-4-3-2-1 sensory exercise."
                        ]
                    }
                }
            ]
        }, f)

persist_dir = "./chroma_health_db"

with open(file_path, "r") as f:
    raw_data = json.load(f)

# Flatten hierarchical JSON
docs = []
for topic_data in raw_data.get("knowledge_base", []):
    topic_name = topic_data.get("topic", "")
    for subtopic_name, content in topic_data.get("subtopics", {}).items():
        clean_subtopic = subtopic_name.replace("_", " ").title()
        content_str = "\n- " + "\n- ".join(content) if isinstance(content, list) else str(content)
        chunk = f"Topic: {topic_name}\nSubtopic: {clean_subtopic}\nInformation: {content_str}"
        docs.append(chunk)

vectorstore = Chroma.from_texts(
    texts=docs,
    embedding=embeddings,
    collection_name="mental_health_kb",
    persist_directory=persist_dir
)
print(f"✅ Persistent RAG vector store ready! Saved to {persist_dir}")

⏳ Setting up Persistent Knowledge Base...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Persistent RAG vector store ready! Saved to ./chroma_health_db


# Reranking Retriever

In [12]:
# ==========================================
# 4. RERANKING RETRIEVER
# ==========================================
print("⏳ Loading Cross-Encoder Reranker...")
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
cross_encoder_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=cross_encoder_model, top_n=2)
retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=base_retriever)
print("✅ Advanced Reranking Pipeline Ready!")

⏳ Loading Cross-Encoder Reranker...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Advanced Reranking Pipeline Ready!


# Emotion Analysis Model

In [13]:
# ==========================================
# 5. EMOTION ANALYSIS MODEL
# ==========================================
print("⏳ Loading Emotion Classification Model...")
emotion_pipeline = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=False)
print("✅ Emotion analysis agent ready!")

⏳ Loading Emotion Classification Model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Emotion analysis agent ready!


# State Defination

In [14]:
# ==========================================
# 6. STATE DEFINITION
# ==========================================
class ConversationState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    user_input: str
    intent: str
    sentiment: str
    is_crisis: bool
    retrieved_knowledge: str
    coping_strategies: str
    final_response: str

# Agent Definitions

In [15]:
# ==========================================
# 7. AGENT DEFINITIONS (NODES)
# ==========================================
def router_agent(state: ConversationState) -> ConversationState:
    """Context-aware router that remembers previous messages."""
    user_input = state["user_input"]
    context = ""
    if len(state["messages"]) > 1:
        context = f"\nPrevious Bot Message: {state['messages'][-2].content}\n"

    prompt = ChatPromptTemplate.from_template(
        "Classify the user's input intent into EXACTLY ONE of the following categories:\n"
        "- 'greeting': Simple hellos, how are you.\n"
        "- 'mental_health_query': Questions about anxiety, stress, therapy, or FOLLOW-UPS to the previous message.\n"
        "- 'crisis': Severe distress, self-harm, emergencies.\n"
        "- 'off_topic': Coding, math, trivia, general non-health topics.\n\n"
        "If the user is asking a follow-up question to the Previous Bot Message, classify it as 'mental_health_query'.\n"
        "{context}"
        "User message: {user_input}\n\nRespond with JUST the category word."
    )
    response = (prompt | llm).invoke({"context": context, "user_input": user_input})
    intent_raw = response.content.strip().lower()

    intent = "mental_health_query"
    for valid in ["greeting", "mental_health_query", "crisis", "off_topic"]:
        if valid in intent_raw: intent = valid; break
    return {"intent": intent}

def off_topic_handler_agent(state: ConversationState) -> ConversationState:
    response = "I am a specialized mental health assistant. I'm not equipped to answer general trivia or off-topic questions. However, if you'd like to talk about how you're feeling or manage stress, I'm here for you!"
    return {"final_response": response, "messages": [AIMessage(content=response)]}

def sentiment_analyzer_agent(state: ConversationState) -> ConversationState:
    user_input = state["user_input"]
    try:
        result = emotion_pipeline(user_input)[0]
        sentiment_map = {"joy": "positive", "surprise": "neutral", "neutral": "neutral", "sadness": "distressed", "fear": "distressed", "anger": "negative", "disgust": "negative"}
        sentiment = sentiment_map.get(result['label'], "neutral")
    except:
        sentiment = "neutral"
    return {"sentiment": sentiment}

def crisis_detector_agent(state: ConversationState) -> ConversationState:
    prompt = ChatPromptTemplate.from_template("Detect if this message indicates a mental health crisis requiring immediate help (suicide, self-harm, urgent danger):\nUser message: {user_input}\nRespond with: yes or no")
    response = (prompt | llm).invoke({"user_input": state["user_input"]})
    return {"is_crisis": "yes" in response.content.strip().lower()}

def knowledge_retrieval_agent(state: ConversationState) -> ConversationState:
    """Context-aware retriever that combines history for accurate searching."""
    user_input = state["user_input"]
    search_query = user_input

    if len(state["messages"]) > 1 and len(user_input.split()) < 8:
        search_query = f"Context: {state['messages'][-2].content[:200]}... Query: {user_input}"

    try:
        relevant_docs = retriever.invoke(search_query)
        retrieved_knowledge = "\n\n".join([doc.page_content[:500] for doc in relevant_docs])
        return {"retrieved_knowledge": retrieved_knowledge}
    except Exception as e:
        return {"retrieved_knowledge": "General mental health support information available."}

def counselor_agent(state: ConversationState) -> ConversationState:
    """Memory-rich counselor that reads the transcript."""
    history_text = ""
    for msg in state["messages"][:-1][-6:]:
        role = "👤 User" if msg.type == "human" else "🤖 Counselor"
        history_text += f"{role}: {msg.content}\n"

    prompt = ChatPromptTemplate.from_template(
        "You are an empathetic mental health counselor.\n"
        "--- Conversation History ---\n{history}\n--------------------------\n\n"
        "Current User: {user_input}\nRelevant Knowledge: {retrieved_knowledge}\n\n"
        "Provide warm, supportive guidance (3-4 sentences max) directly answering the current user prompt while using the history for context. Do NOT diagnose."
    )
    response = (prompt | llm).invoke({
        "history": history_text,
        "user_input": state["user_input"],
        "retrieved_knowledge": state.get("retrieved_knowledge", "")
    })
    return {"final_response": response.content.strip()}

def coping_strategy_agent(state: ConversationState) -> ConversationState:
    prompt = ChatPromptTemplate.from_template("Based on this information, suggest 2-3 practical coping strategies:\n{retrieved_knowledge}\nFormat as bullet points. Keep each strategy to one sentence.")
    response = (prompt | llm).invoke({"retrieved_knowledge": state.get("retrieved_knowledge", "")})
    return {"coping_strategies": response.content.strip()}

def response_formatter_agent(state: ConversationState) -> ConversationState:
    counselor_response = state.get("final_response", "")
    coping_strategies = state.get("coping_strategies", "")

    if coping_strategies and state.get("intent") != "greeting":
        formatted_response = f"{counselor_response}\n\n**Practical strategies you might try:**\n{coping_strategies}\n"
    else:
        formatted_response = counselor_response

    return {"final_response": formatted_response, "messages": [AIMessage(content=formatted_response)]}

def crisis_handler_agent(state: ConversationState) -> ConversationState:
    crisis_response = (
        "🚨 **CRISIS SUPPORT - IMMEDIATE HELP NEEDED** 🚨\n\n"
        "I'm concerned about what you're experiencing. Please reach out for immediate professional help:\n\n"
        "📞 **Emergency Resources:**\n"
        "• National Suicide Prevention Lifeline: **988** (24/7, free, confidential)\n"
        "• India: iCall **9152987821** or dial **112**\n"
        "• Crisis Text Line: Text **HOME** to **741741**\n"
        "• International: https://findahelpline.com\n\n"
        "You deserve support, and help is available right now. Please reach out immediately."
    )
    return {"final_response": crisis_response, "messages": [AIMessage(content=crisis_response)]}

# Build the Granph & Conditional Routhing

In [16]:
# ==========================================
# 8. BUILD THE GRAPH & CONDITIONAL ROUTING
# ==========================================
def route_from_router(state: ConversationState):
    if state.get("intent") == "off_topic": return "off_topic_handler"
    return "sentiment"

def route_crisis(state: ConversationState):
    if state.get("is_crisis"): return "crisis_handler"
    return "retrieve_knowledge"

workflow = StateGraph(ConversationState)

workflow.add_node("router", router_agent)
workflow.add_node("off_topic_handler", off_topic_handler_agent)
workflow.add_node("sentiment", sentiment_analyzer_agent)
workflow.add_node("crisis_detector", crisis_detector_agent)
workflow.add_node("crisis_handler", crisis_handler_agent)
workflow.add_node("retrieve_knowledge", knowledge_retrieval_agent)
workflow.add_node("counselor", counselor_agent)
workflow.add_node("coping", coping_strategy_agent)
workflow.add_node("formatter", response_formatter_agent)

workflow.add_edge(START, "router")
workflow.add_conditional_edges("router", route_from_router, {"off_topic_handler": "off_topic_handler", "sentiment": "sentiment"})
workflow.add_edge("off_topic_handler", END)

workflow.add_edge("sentiment", "crisis_detector")
workflow.add_conditional_edges("crisis_detector", route_crisis, {"crisis_handler": "crisis_handler", "retrieve_knowledge": "retrieve_knowledge"})

workflow.add_edge("crisis_handler", END)
workflow.add_edge("retrieve_knowledge", "counselor")
workflow.add_edge("counselor", "coping")
workflow.add_edge("coping", "formatter")
workflow.add_edge("formatter", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)
print("✅ Agentic workflow compiled with Memory & Guardrails!")

✅ Agentic workflow compiled with Memory & Guardrails!


# Continous Chat Loop

In [17]:
# ==========================================
# 9. CONTINUOUS CHAT LOOP
# ==========================================
# We use a static thread_id for this notebook session so it remembers the chat
config = {"configurable": {"thread_id": "notebook_session_01"}}

print("\n" + "="*60)
print("🤖 Mental Health Chatbot Initialized!")
print("Type 'quit' or 'exit' to stop. Type 'clear' to reset memory.")
print("="*60 + "\n")

while True:
    try:
        user_input = input("👤 You: ")

        if user_input.lower() in ['quit', 'exit']:
            print("\n🤖 Bot: Take care! Shutting down...")
            break

        if user_input.lower() == 'clear':
            # Changing the thread ID starts a completely fresh memory state
            config["configurable"]["thread_id"] += "_new"
            print("\n🧹 Memory cleared! Starting a fresh conversation.\n")
            continue

        if not user_input.strip():
            continue

        # We inject the new human message into the state
        initial_state = {
            "user_input": user_input,
            "messages": [HumanMessage(content=user_input)]
        }

        result = app.invoke(initial_state, config=config)
        print(f"\n🤖 Bot:\n{result['final_response']}\n")
        print("-" * 60)

    except KeyboardInterrupt:
        print("\n🤖 Bot: Take care! Shutting down...\n*Remember: You're not alone, and it's okay to ask for help. 💙*")
        break
    except Exception as e:
        print(f"\n❌ An error occurred: {e}")


🤖 Mental Health Chatbot Initialized!
Type 'quit' or 'exit' to stop. Type 'clear' to reset memory.

👤 You: How to get reed to Depression?

🤖 Bot:
It can help to build a gentle daily routine—wake, eat, and sleep at roughly the same times to create a sense of stability. Try short walks or simple stretches and get a few minutes of natural sunlight each day, as both movement and light can lift mood. Keep up basic self‑care: a shower, fresh clothes, and a quick tidy‑up can feel surprisingly grounding. Reach out to a trusted friend or family member and share what you’re feeling; connecting with others and doing small activities you enjoy can counter isolation and bring a bit of joy back into your day.

**Practical strategies you might try:**
- Create a simple daily schedule that includes a short walk, a shower, and time for a favorite song to bring structure and purpose.  
- Step outside for at least 10 minutes each day to soak up natural light, boosting vitamin D and mood regulation.  
- Re